# Knowledge Graph Exploration

This notebook loads and explores the knowledge graph generated by `run_extraction.py`.

In [ ]:
import os
import sys
import networkx as nx
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, BASE_DIR)

from src.inferencer import GraphInferencer
from src.querier import GraphQuerier

GRAPH_PATH = os.path.join(BASE_DIR, 'data', 'graphs', 'knowledge_graph.graphml')

if not os.path.exists(GRAPH_PATH):
    raise FileNotFoundError(f'Graph not found: {GRAPH_PATH}. Run run_extraction.py first.')

querier = GraphQuerier(GRAPH_PATH)
G = querier.graph
inferencer = GraphInferencer(G)

print(f'Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges')

## Basic Statistics

In [ ]:
entity_types = querier.get_all_entity_types()
relation_types = querier.get_all_relation_types()

print(f'Entity types ({len(entity_types)}): {entity_types}')
print(f'Relation types ({len(relation_types)}): {relation_types}')

print('\nEntity count by type:')
for t, c in sorted(querier.get_entity_count_by_type().items()):
    print(f'  {t}: {c}')

print('\nRelation count by type:')
for t, c in sorted(querier.get_relation_count_by_type().items()):
    print(f'  {t}: {c}')

## Centrality Analysis

In [ ]:
centrality = inferencer.compute_centrality()
df_centrality = pd.DataFrame(centrality).head(10)
display(df_centrality)

## Community Detection

In [ ]:
communities = inferencer.find_communities()
print(f'Found {len(communities)} communities')

sizes = [c['size'] for c in communities]
plt.bar(range(len(sizes)), sizes)
plt.xlabel('Community ID')
plt.ylabel('Size')
plt.title('Community Sizes')
plt.show()

## Query Examples

In [ ]:
# Find all relations for a specific entity
entity_name = list(G.nodes())[0] if G.number_of_nodes() > 0 else None
if entity_name:
    print(f'Connections for: {entity_name}')
    connections = querier.find_connections(entity_name, depth=2)
    for conn in connections[:10]:
        print(f"  {conn['entity']} (distance: {conn['distance']})")

## Visualization

In [ ]:
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, k=0.3, seed=42)
nx.draw_networkx(G, pos, node_size=500, font_size=8, with_labels=True, node_color='lightblue', edge_color='gray')
plt.title('Knowledge Graph')
plt.axis('off')
plt.show()

## Export Cypher for Neo4j

In [ ]:
cypher = querier.cypher_style_export()
print(cypher[:1000])
if len(cypher) > 1000:
    print('... (truncated)')